# 53. DoRA：怎样把权重方向的低秩更新与行幅值分开学习？

## 面试回答主线

LoRA 直接学习低秩权重增量 `BA`，而 DoRA 将每个输出行拆成方向与 magnitude：低秩分支主要改变方向，独立参数学习行范数。forward 时先计算 `W0 + BA`，按行归一化，再乘可训练 magnitude；基座权重保持冻结。这个分解让低秩方向无法表达的逐行尺度变化拥有单独自由度。面试时我会实现完整 `nn.Module`、运行真实梯度训练，并在同一批工单路由样本上比较冻结基座、同 rank LoRA 与 DoRA。还要解释初始化：若 A、B 同时为零，乘积两边梯度都为零，adapter 会彻底不学习。生产实现需处理 merge、量化、fan-in/fan-out、epsilon 和 checkpoint 格式，而不是只写一行公式。

## 1. 真实案例：六条客服工单映射到优先级与人工升级分数

输入三个特征依次是紧急度、负面情绪和金额强度；输出两个连续分数用于“处理优先级”和“人工升级”。目标域同时改变两个输出行的方向与幅值，正适合观察 DoRA 的分解。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示适配数据
import torch  # 导入 PyTorch 实现真实线性层 forward 与梯度训练
from torch import nn  # 导入神经网络基类定义 LoRA 和 DoRA 模块
from torch.nn import functional as F  # 导入线性运算与均方误差损失
torch.set_num_threads(1)  # 限制教学训练线程数以保持稳定运行
torch.manual_seed(23)  # 固定低秩参数初始化和训练结果
base_weight = torch.tensor([[0.8, -0.3, 0.2], [-0.2, 0.7, 0.4]], dtype=torch.float32)  # 定义冻结基座模型的二乘三权重
true_A = torch.tensor([[0.5, -0.4, 0.3]], dtype=torch.float32)  # 定义目标域真实的 rank-1 方向变化
true_B = torch.tensor([[0.6], [-0.5]], dtype=torch.float32)  # 定义目标域真实的 rank-1 输出投影
target_direction = base_weight + true_B @ true_A  # 构造 DoRA 可以表示的目标方向
target_magnitude = torch.tensor([[1.8], [0.65]], dtype=torch.float32)  # 构造两个输出行独立的目标幅值
target_weight = target_magnitude * target_direction / target_direction.norm(dim=1, keepdim=True)  # 按行归一化方向并施加目标幅值
cases = [{"id": "R01", "text": "普通咨询，低金额", "features": [1.0, 0.0, 0.2]}, {"id": "R02", "text": "催促退款，轻微不满", "features": [0.8, 0.2, 0.1]}, {"id": "R03", "text": "投诉延误，中等金额", "features": [0.2, 1.0, 0.3]}, {"id": "R04", "text": "多次投诉，需要升级", "features": [0.1, 0.8, 0.5]}, {"id": "R05", "text": "紧急且明显不满", "features": [0.7, 0.7, 0.2]}, {"id": "R06", "text": "高金额争议", "features": [0.4, 0.3, 1.0]}]  # 定义六条具有真实路由语义的工单
features = torch.tensor([item["features"] for item in cases], dtype=torch.float32)  # 把六条工单特征组成训练矩阵
targets = F.linear(features, target_weight)  # 用目标域权重生成可复现的双输出监督值
preview = [{"工单": item["id"], "描述": item["text"], "特征": item["features"], "目标输出": [round(value, 3) for value in targets[index].tolist()]} for index, item in enumerate(cases)]  # 汇总业务输入与目标分数
print("DoRA 适配案例预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示六条工单及其双输出监督

DoRA 适配案例预览：
[{'工单': 'R01', '描述': '普通咨询，低金额', '特征': [1.0, 0.0, 0.2], '目标输出': [1.65, -0.251]},
 {'工单': 'R02',
  '描述': '催促退款，轻微不满',
  '特征': [0.8, 0.2, 0.1],
  '目标输出': [1.136, -0.097]},
 {'工单': 'R03',
  '描述': '投诉延误，中等金额',
  '特征': [0.2, 1.0, 0.3],
  '目标输出': [-0.289, 0.555]},
 {'工单': 'R04',
  '描述': '多次投诉，需要升级',
  '特征': [0.1, 0.8, 0.5],
  '目标输出': [-0.185, 0.502]},
 {'工单': 'R05', '描述': '紧急且明显不满', '特征': [0.7, 0.7, 0.2], '目标输出': [0.657, 0.229]},
 {'工单': 'R06', '描述': '高金额争议', '特征': [0.4, 0.3, 1.0], '目标输出': [0.923, 0.213]}]


## 2. Baseline（基线）：冻结基座权重直接服务新领域

基座模型没有看过目标域的幅值与方向变化。下面在相同六条工单上执行真实 `F.linear`，逐条输出预测与绝对误差；它是后续 LoRA、DoRA 的共同起点。

In [2]:
with torch.no_grad():  # 在不建立梯度图的情况下评估冻结基座
    base_predictions = F.linear(features, base_weight)  # 用原始权重对六条工单执行线性 forward
base_rows = []  # 收集冻结基座的逐工单误差
for index, item in enumerate(cases):  # 遍历全部真实工单样本
    absolute_error = torch.abs(base_predictions[index] - targets[index]).mean()  # 计算当前工单两个输出的平均绝对误差
    base_rows.append({"工单": item["id"], "基座预测": [round(value, 3) for value in base_predictions[index].tolist()], "目标": [round(value, 3) for value in targets[index].tolist()], "平均绝对误差": round(float(absolute_error), 4)})  # 保存逐样本基线结果
base_mse = float(F.mse_loss(base_predictions, targets))  # 汇总冻结基座在目标域的均方误差
print("冻结基座在目标域的逐工单结果：")  # 标注当前输出属于基线
pprint(base_rows, sort_dicts=False)  # 展示每条业务输入的真实预测偏差
print(f"冻结基座 MSE={base_mse:.6f}")  # 输出后续适配方案的共同比较基准

冻结基座在目标域的逐工单结果：
[{'工单': 'R01', '基座预测': [0.84, -0.12], '目标': [1.65, -0.251], '平均绝对误差': 0.4703},
 {'工单': 'R02', '基座预测': [0.6, 0.02], '目标': [1.136, -0.097], '平均绝对误差': 0.3268},
 {'工单': 'R03', '基座预测': [-0.08, 0.78], '目标': [-0.289, 0.555], '平均绝对误差': 0.2171},
 {'工单': 'R04', '基座预测': [-0.06, 0.74], '目标': [-0.185, 0.502], '平均绝对误差': 0.1818},
 {'工单': 'R05', '基座预测': [0.39, 0.43], '目标': [0.657, 0.229], '平均绝对误差': 0.2339},
 {'工单': 'R06', '基座预测': [0.43, 0.53], '目标': [0.923, 0.213], '平均绝对误差': 0.405}]
冻结基座 MSE=0.133056


## 3. 手写 PyTorch 架构：同 rank LoRA 与 DoRA forward

两者都冻结 `base_weight` 并使用 rank=1 的 A、B。LoRA 直接相加；DoRA 对方向矩阵逐行归一化后乘独立 magnitude。A 随机小初始化、B 零初始化，使首步 B 有梯度且初始模型仍等于基座。

In [3]:
class LoRALinear(nn.Module):  # 定义直接叠加低秩增量的 LoRA 线性层
    def __init__(self, frozen_weight, rank):  # 初始化冻结基座与低秩因子
        super().__init__()  # 注册 PyTorch 模块状态
        self.register_buffer("base", frozen_weight.clone())  # 把基座权重注册为不参与训练的 buffer
        self.A = nn.Parameter(torch.randn(rank, frozen_weight.shape[1]) * 0.02)  # 随机小初始化输入低秩因子以打破对称
        self.B = nn.Parameter(torch.zeros(frozen_weight.shape[0], rank))  # 零初始化输出因子保持初始权重等于基座
    def adapted_weight(self):  # 计算 LoRA 合并后的有效权重
        return self.base + self.B @ self.A  # 直接把 rank-1 增量加到冻结基座
    def forward(self, inputs):  # 对一批业务特征执行 LoRA 线性 forward
        return F.linear(inputs, self.adapted_weight())  # 使用实际合并权重计算双输出
class DoRALinear(nn.Module):  # 定义权重幅值与方向分解的 DoRA 线性层
    def __init__(self, frozen_weight, rank):  # 初始化冻结方向、低秩因子与可训练幅值
        super().__init__()  # 注册 PyTorch 模块状态
        self.register_buffer("base", frozen_weight.clone())  # 保存不参与优化的基座权重
        self.A = nn.Parameter(torch.randn(rank, frozen_weight.shape[1]) * 0.02)  # 随机小初始化方向更新的输入因子
        self.B = nn.Parameter(torch.zeros(frozen_weight.shape[0], rank))  # 零初始化输出因子保持初始方向不变
        self.magnitude = nn.Parameter(frozen_weight.norm(dim=1, keepdim=True))  # 用基座每行范数初始化独立幅值
    def adapted_weight(self):  # 根据低秩方向和行幅值计算 DoRA 有效权重
        direction = self.base + self.B @ self.A  # 先得到带 rank-1 更新的未归一化方向
        unit_direction = direction / direction.norm(dim=1, keepdim=True).clamp_min(1e-8)  # 按输出行归一化并防止除零
        return self.magnitude * unit_direction  # 用独立可训练幅值重新缩放每个输出方向
    def forward(self, inputs):  # 对一批业务特征执行 DoRA 线性 forward
        return F.linear(inputs, self.adapted_weight())  # 使用分解后有效权重计算双输出
torch.manual_seed(23)  # 为 LoRA 初始化固定公平起点
lora_model = LoRALinear(base_weight, rank=1)  # 实例化 rank-1 LoRA 对照模型
torch.manual_seed(23)  # 为 DoRA 复用同一低秩随机初始化
dora_model = DoRALinear(base_weight, rank=1)  # 实例化 rank-1 DoRA 目标模型
print({"LoRA可训练参数": sum(parameter.numel() for parameter in lora_model.parameters()), "DoRA可训练参数": sum(parameter.numel() for parameter in dora_model.parameters()), "基座行范数": base_weight.norm(dim=1).tolist(), "DoRA初始幅值": dora_model.magnitude.detach().flatten().tolist()})  # 展示架构参数和幅值初始化

{'LoRA可训练参数': 5, 'DoRA可训练参数': 7, '基座行范数': [0.8774964809417725, 0.8306623697280884], 'DoRA初始幅值': [0.8774964809417725, 0.8306623697280884]}


## 4. 真实梯度训练：比较损失曲线、梯度与行范数

两种 adapter 使用相同数据、rank、优化器和训练步数。DoRA 多出的 magnitude 参数允许独立匹配两个输出行的目标范数；下面记录首步 B 梯度、初末损失和训练后权重行范数。

In [4]:
def fit_adapter(model, steps=1200):  # 用相同配置训练一个低秩适配器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)  # 创建更新 adapter 参数的优化器
    history = []  # 记录每一步目标域均方误差
    first_B_gradient = 0.0  # 初始化首步输出因子梯度范数
    for step in range(steps):  # 在六条目标域工单上执行真实梯度训练
        predictions = model(features)  # 通过 LoRA 或 DoRA forward 计算双输出
        loss = F.mse_loss(predictions, targets)  # 计算与目标域监督的均方误差
        optimizer.zero_grad()  # 清除上一轮 adapter 梯度
        loss.backward()  # 通过低秩因子和 DoRA 幅值执行反向传播
        if step == 0:  # 在第一次参数更新前记录梯度证据
            first_B_gradient = float(model.B.grad.norm())  # 读取零初始化 B 是否收到非零梯度
        optimizer.step()  # 应用当前梯度更新 adapter 参数
        history.append(float(loss.detach()))  # 保存当前损失用于收敛对照
    return history, first_B_gradient  # 返回完整训练轨迹与首步梯度
lora_history, lora_first_gradient = fit_adapter(lora_model)  # 训练 rank-1 LoRA 对照模型
dora_history, dora_first_gradient = fit_adapter(dora_model)  # 训练 rank-1 DoRA 幅值方向分解模型
with torch.no_grad():  # 在不建立梯度图的情况下读取最终模型指标
    lora_predictions = lora_model(features)  # 计算 LoRA 在六条工单上的最终输出
    dora_predictions = dora_model(features)  # 计算 DoRA 在六条工单上的最终输出
    lora_mse = float(F.mse_loss(lora_predictions, targets))  # 汇总 LoRA 最终均方误差
    dora_mse = float(F.mse_loss(dora_predictions, targets))  # 汇总 DoRA 最终均方误差
print({"LoRA": {"首步B梯度": round(lora_first_gradient, 6), "初始loss": round(lora_history[0], 6), "最终MSE": round(lora_mse, 8), "行范数": [round(value, 4) for value in lora_model.adapted_weight().detach().norm(dim=1).tolist()]}, "DoRA": {"首步B梯度": round(dora_first_gradient, 6), "初始loss": round(dora_history[0], 6), "最终MSE": round(dora_mse, 10), "行范数": [round(value, 4) for value in dora_model.adapted_weight().detach().norm(dim=1).tolist()]}, "目标行范数": [round(value, 4) for value in target_weight.norm(dim=1).tolist()]})  # 展示真实梯度、收敛与幅值拟合

{'LoRA': {'首步B梯度': 0.005635, '初始loss': 0.133056, '最终MSE': 0.01649465, '行范数': [1.7599, 0.9451]}, 'DoRA': {'首步B梯度': 0.002412, '初始loss': 0.133056, '最终MSE': 0.0004839233, '行范数': [1.7993, 0.6439]}, '目标行范数': [1.8, 0.65]}


## 5. 结果解读：逐工单比较 LoRA 与 DoRA 输出误差

目标权重由“rank-1 方向变化 + 独立行幅值”生成，因此 DoRA 表达假设与数据一致；LoRA rank-1 需要同时用一个增量承担方向和尺度，留下不可约误差。这个受控结果说明机制差异，不代表 DoRA 在所有真实模型上必胜。

In [5]:
result_rows = []  # 收集同一批工单上的三种模型结果
for index, item in enumerate(cases):  # 遍历六条真实目标域输入
    lora_error = float(torch.abs(lora_predictions[index] - targets[index]).mean())  # 计算 LoRA 当前工单平均绝对误差
    dora_error = float(torch.abs(dora_predictions[index] - targets[index]).mean())  # 计算 DoRA 当前工单平均绝对误差
    result_rows.append({"工单": item["id"], "目标": [round(value, 3) for value in targets[index].tolist()], "LoRA": [round(value, 3) for value in lora_predictions[index].tolist()], "DoRA": [round(value, 3) for value in dora_predictions[index].tolist()], "LoRA误差": round(lora_error, 5), "DoRA误差": round(dora_error, 5)})  # 保存逐样本可解释对照
print("LoRA 与 DoRA 逐工单适配结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示幅值方向分解在每条工单上的影响
print(f"总体 MSE：冻结基座={base_mse:.6f}，LoRA={lora_mse:.6f}，DoRA={dora_mse:.10f}")  # 汇总三种方案在同数据上的拟合误差

LoRA 与 DoRA 逐工单适配结果：
[{'工单': 'R01',
  '目标': [1.65, -0.251],
  'LoRA': [1.628, -0.332],
  'DoRA': [1.651, -0.253],
  'LoRA误差': 0.0518,
  'DoRA误差': 0.00122},
 {'工单': 'R02',
  '目标': [1.136, -0.097],
  'LoRA': [1.13, -0.123],
  'DoRA': [1.138, -0.113],
  'LoRA误差': 0.01622,
  'DoRA误差': 0.00866},
 {'工单': 'R03',
  '目标': [-0.289, 0.555],
  'LoRA': [-0.218, 0.817],
  'DoRA': [-0.288, 0.528],
  'LoRA误差': 0.1666,
  'DoRA误差': 0.01373},
 {'工单': 'R04',
  '目标': [-0.185, 0.502],
  'LoRA': [-0.117, 0.755],
  'DoRA': [-0.186, 0.503],
  'LoRA误差': 0.16109,
  'DoRA误差': 0.00119},
 {'工单': 'R05',
  '目标': [0.657, 0.229],
  'LoRA': [0.689, 0.349],
  'DoRA': [0.658, 0.199],
  'LoRA误差': 0.07653,
  'DoRA误差': 0.01573},
 {'工单': 'R06',
  '目标': [0.923, 0.213],
  'LoRA': [0.969, 0.385],
  'DoRA': [0.92, 0.275],
  'LoRA误差': 0.10884,
  'DoRA误差': 0.03293}]
总体 MSE：冻结基座=0.133056，LoRA=0.016495，DoRA=0.0004839233


## 6. 失败案例与修正：A、B 同时零初始化造成 dead adapter

对乘积 `BA`，B 的梯度包含 A，A 的梯度包含 B；两者同时为零时首步两个梯度都为零，之后永远无法离开原点。正确初始化是一边随机小值、另一边为零：初始函数仍等于基座，但零侧在第一步能收到梯度。下面用真实 backward 复现。

In [6]:
dead_A = torch.zeros(1, 3, requires_grad=True)  # 构造错误的全零输入低秩因子
dead_B = torch.zeros(2, 1, requires_grad=True)  # 构造错误的全零输出低秩因子
dead_predictions = F.linear(features, base_weight + dead_B @ dead_A)  # 用双零 adapter 执行实际 forward
dead_loss = F.mse_loss(dead_predictions, targets)  # 计算双零初始化下仍然存在的目标域误差
dead_loss.backward()  # 反向传播检查两个低秩因子的真实梯度
dead_gradients = (float(dead_A.grad.norm()), float(dead_B.grad.norm()))  # 读取两侧都无法启动的零梯度
fixed_A = (torch.randn(1, 3) * 0.02).requires_grad_()  # 用随机小值初始化输入因子打破乘法对称
fixed_B = torch.zeros(2, 1, requires_grad=True)  # 保持输出因子为零以维持初始基座函数
fixed_predictions = F.linear(features, base_weight + fixed_B @ fixed_A)  # 用一随机一零的 adapter 执行 forward
fixed_loss = F.mse_loss(fixed_predictions, targets)  # 计算修正初始化下的同一目标域误差
fixed_loss.backward()  # 反向传播验证零侧 B 能在首步得到学习信号
fixed_gradients = (float(fixed_A.grad.norm()), float(fixed_B.grad.norm()))  # 读取修正后两侧首步梯度状态
print({"失败_双零梯度(A,B)": dead_gradients, "修正_一随机一零梯度(A,B)": fixed_gradients, "解释": "首步B获得非零梯度，下一步A随后获得梯度"})  # 展示 dead adapter 根因和初始化修正

{'失败_双零梯度(A,B)': (0.0, 0.0), '修正_一随机一零梯度(A,B)': (0.0, 0.007368069142103195), '解释': '首步B获得非零梯度，下一步A随后获得梯度'}


## 7. 生产差距与最小回归检查

真实 Transformer 权重可能采用转置布局，DoRA norm 维度必须与 Linear 的输出行定义一致。量化基座、FSDP 分片、adapter merge/unmerge、dropout 与 epsilon 都会影响数值；保存时还要同时版本化 A、B、magnitude 和基座 checkpoint。评测应覆盖训练集外任务、显存、吞吐及合并前后输出一致性。最后的断言只验证本实验的 forward、梯度、拟合差异与双零失败。

In [7]:
assert len(cases) >= 5  # 确认真实工单样本数量满足逐样本教学要求
assert lora_first_gradient > 0.0 and dora_first_gradient > 0.0  # 确认两种 adapter 都通过真实反向传播获得学习信号
assert lora_history[-1] < lora_history[0]  # 确认 LoRA 训练过程真实降低目标域损失
assert dora_history[-1] < dora_history[0]  # 确认 DoRA 训练过程真实降低目标域损失
assert dora_mse < lora_mse  # 确认幅值方向分解在受控目标权重上优于同 rank LoRA
assert dora_mse < 1e-3  # 确认 DoRA 将由其参数化生成的目标映射误差压到千分之一以下
assert dead_gradients == (0.0, 0.0) and fixed_gradients[1] > 0.0  # 确认双零 dead adapter 与修正初始化均被真实复现
print("回归检查通过：DoRA forward、梯度训练、幅值拟合与初始化失败均已验证。")  # 输出最终验收结论

回归检查通过：DoRA forward、梯度训练、幅值拟合与初始化失败均已验证。
